# DriveGuard - M3 backfill: N=7 and N=15 horizons

Builds rolling features for the shorter failure horizons (7 and 15 days) and benchmarks the
top tree models. LightGBM uses the Optuna-tuned config from N=30 (config/best_params.json);
RF/CatBoost use defaults. Confirms the approach holds across horizons.

**Settings:** Add data `driveguard-backblaze-interim`; GPU on (CatBoost); Internet On;
Secret `GITHUB_TOKEN`.

In [ ]:
# 1. Clone repo
import os, subprocess
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
URL = f'https://{token}@github.com/keerthirevanth/driveguard-predictive-maintenance.git'
REPO = '/kaggle/working/driveguard-predictive-maintenance'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', URL, REPO], check=True)
import sys; sys.path.insert(0, f'{REPO}/src')
print('cloned:', os.path.exists(REPO))

In [ ]:
# 2. Install deps
!pip install -q polars pyarrow mlflow catboost 2>/dev/null
print('deps installed')

In [ ]:
# 3. Link dataset
import os, glob
os.makedirs(f'{REPO}/data/interim', exist_ok=True)
os.makedirs(f'{REPO}/data/processed', exist_ok=True)
qf = glob.glob('/kaggle/input/**/data_*.parquet', recursive=True)
sf = glob.glob('/kaggle/input/**/drive_summary.parquet', recursive=True)
assert qf and sf, 'dataset not attached?'
def link(src, dst):
    if os.path.lexists(dst): os.remove(dst)
    os.symlink(src, dst)
for f in qf: link(f, f"{REPO}/data/interim/{os.path.basename(f)}")
link(sf[0], f'{REPO}/data/processed/drive_summary.parquet')
print('interim:', sorted(os.listdir(f'{REPO}/data/interim')))

In [ ]:
# 4. Build rolling features + bake-off for N=7 and N=15
import json
from pathlib import Path
from driveguard.config import load_config
from driveguard.features.rolling import make_rolling_dataset
from driveguard.models.train import run_bakeoff
ROOT = Path(REPO); cfg = load_config(f'{REPO}/config/config.yaml')
best = json.load(open(f'{REPO}/config/best_params.json'))['params']
MODEL_PARAMS = {'lightgbm': best['lightgbm']}  # tuned LightGBM; RF/CatBoost use defaults
MODELS = ['lightgbm', 'random_forest', 'catboost']
all_results = []
for N in [7, 15]:
    print(f'=== building rolling features N={N} ==='); make_rolling_dataset(cfg, ROOT, N)
    fdir = f'{REPO}/data/processed/features_rolling_N{N}'
    board = run_bakeoff(fdir, 'rolling', N, MODELS, cfg,
                        mlflow_uri='/kaggle/working/mlruns', model_params=MODEL_PARAMS)
    all_results += board
    json.dump(board, open(f'/kaggle/working/bakeoff_rolling_N{N}.json', 'w'), indent=2)
print('horizons backfill complete')

In [ ]:
# 5. Leaderboard across horizons
import pandas as pd
rows = []
for r in all_results:
    if r.get('status') == 'ok':
        rows.append({'horizon': r['horizon'], 'model': r['model'],
                     'test_pr_auc': round(r['test']['pr_auc'], 4),
                     'test_recall@1%fpr': round((r['test']['recall_at_fpr_1pct']['recall'] or 0), 3)})
    else:
        rows.append({'horizon': r.get('horizon'), 'model': r['model'], 'test_pr_auc': 'ERR'})
print('For reference, N=30 tuned LightGBM test PR-AUC = 0.164')
pd.DataFrame(rows).sort_values(['horizon', 'test_pr_auc'], ascending=[True, False])

In [ ]:
# 6. Package artifacts
import shutil, os
if os.path.isdir('/kaggle/working/mlruns'):
    shutil.make_archive('/kaggle/working/mlruns_export', 'zip',
                        root_dir='/kaggle/working', base_dir='mlruns')
!ls -lh /kaggle/working/*.json